In [1]:
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!pip install -q pyspark

In [2]:
import seaborn as sns
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

In [3]:
spark = (
    SparkSession.builder
    .appName("PySpark_Tips_Ejercicios")
    .getOrCreate()
)

In [4]:
tips_pd = sns.load_dataset("tips")
tips_pd.head()

,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3
3,23.68,3.31,Male,No,Sun,Dinner,2
4,24.59,3.61,Female,No,Sun,Dinner,4


In [5]:
df = spark.createDataFrame(tips_pd)
df.printSchema()
df.show(5)

root
 |-- total_bill: double (nullable = true)
 |-- tip: double (nullable = true)
 |-- sex: string (nullable = true)
 |-- smoker: string (nullable = true)
 |-- day: string (nullable = true)
 |-- time: string (nullable = true)
 |-- size: long (nullable = true)

+----------+----+------+------+---+------+----+
|total_bill| tip|   sex|smoker|day|  time|size|
+----------+----+------+------+---+------+----+
|     16.99|1.01|Female|    No|Sun|Dinner|   2|
|     10.34|1.66|  Male|    No|Sun|Dinner|   3|
|     21.01| 3.5|  Male|    No|Sun|Dinner|   3|
|     23.68|3.31|  Male|    No|Sun|Dinner|   2|
|     24.59|3.61|Female|    No|Sun|Dinner|   4|
+----------+----+------+------+---+------+----+
only showing top 5 rows


# Ejercicio 1: Crear nuevas columnas con withColumn

## Objetivo
Practicar transformaciones básicas con `withColumn`.

## Instrucciones
A partir del dataset `tips`, realiza lo siguiente:

1. Crear una nueva columna llamada `tip_percentage` que calcule el porcentaje de propina:
   
   tip / total_bill * 100

2. Crear una columna llamada `total_per_person` que calcule cuánto pagó en promedio cada persona:

   total_bill / size

3. Mostrar las columnas:
   - total_bill
   - tip
   - size
   - tip_percentage
   - total_per_person

## Pregunta
¿Qué mesas parecen dejar mayor porcentaje de propina?

In [6]:
df_1 = (
    df
    .withColumn("tip_percentage",(F.col("tip") / F.col("total_bill"))* 100)
    .withColumn("total_per_person",F.col("total_bill") / F.col("size"))
    )

df_1.show(5)

+----------+----+------+------+---+------+----+------------------+------------------+
|total_bill| tip|   sex|smoker|day|  time|size|    tip_percentage|  total_per_person|
+----------+----+------+------+---+------+----+------------------+------------------+
|     16.99|1.01|Female|    No|Sun|Dinner|   2|5.9446733372572105|             8.495|
|     10.34|1.66|  Male|    No|Sun|Dinner|   3|16.054158607350097|3.4466666666666668|
|     21.01| 3.5|  Male|    No|Sun|Dinner|   3|16.658733936220845| 7.003333333333334|
|     23.68|3.31|  Male|    No|Sun|Dinner|   2| 13.97804054054054|             11.84|
|     24.59|3.61|Female|    No|Sun|Dinner|   4|14.680764538430255|            6.1475|
+----------+----+------+------+---+------+----+------------------+------------------+
only showing top 5 rows


In [7]:
df_1.select("total_bill","tip","size","tip_percentage","total_per_person").show(10)

+----------+----+----+------------------+------------------+
|total_bill| tip|size|    tip_percentage|  total_per_person|
+----------+----+----+------------------+------------------+
|     16.99|1.01|   2|5.9446733372572105|             8.495|
|     10.34|1.66|   3|16.054158607350097|3.4466666666666668|
|     21.01| 3.5|   3|16.658733936220845| 7.003333333333334|
|     23.68|3.31|   2| 13.97804054054054|             11.84|
|     24.59|3.61|   4|14.680764538430255|            6.1475|
|     25.29|4.71|   4| 18.62396204033215|            6.3225|
|      8.77| 2.0|   2| 22.80501710376283|             4.385|
|     26.88|3.12|   4|11.607142857142858|              6.72|
|     15.04|1.96|   2|13.031914893617023|              7.52|
|     14.78|3.23|   2|21.853856562922868|              7.39|
+----------+----+----+------------------+------------------+
only showing top 10 rows


# la mesa de 3 y 4 dejan mayor cantidad de propina

# Ejercicio 2: Filtrar registros

## Objetivo
Practicar filtros sobre un DataFrame de PySpark.

## Instrucciones
Filtra el dataset para mostrar solo las cuentas que cumplan estas condiciones:

1. `total_bill` mayor a 20
2. `tip` mayor a 3
3. `time` sea igual a `Dinner`

Luego muestra estas columnas:

- total_bill
- tip
- sex
- smoker
- day
- time

## Pregunta
¿Qué características tienen las cuentas más altas con buenas propinas?

In [9]:
df_2 = (
    df
    .filter(F.col("total_bill") > 20)
    .filter(F.col("tip") > 3)
    .filter(F.col("time") == "Dinner")
)
df_2.show(5)

+----------+----+------+------+---+------+----+
|total_bill| tip|   sex|smoker|day|  time|size|
+----------+----+------+------+---+------+----+
|     21.01| 3.5|  Male|    No|Sun|Dinner|   3|
|     23.68|3.31|  Male|    No|Sun|Dinner|   2|
|     24.59|3.61|Female|    No|Sun|Dinner|   4|
|     25.29|4.71|  Male|    No|Sun|Dinner|   4|
|     26.88|3.12|  Male|    No|Sun|Dinner|   4|
+----------+----+------+------+---+------+----+
only showing top 5 rows


In [10]:
df_2.select("total_bill","tip","sex","smoker","day","time").show(10)

+----------+----+------+------+---+------+
|total_bill| tip|   sex|smoker|day|  time|
+----------+----+------+------+---+------+
|     21.01| 3.5|  Male|    No|Sun|Dinner|
|     23.68|3.31|  Male|    No|Sun|Dinner|
|     24.59|3.61|Female|    No|Sun|Dinner|
|     25.29|4.71|  Male|    No|Sun|Dinner|
|     26.88|3.12|  Male|    No|Sun|Dinner|
|     35.26| 5.0|Female|    No|Sun|Dinner|
|     21.58|3.92|  Male|    No|Sun|Dinner|
|     20.65|3.35|  Male|    No|Sat|Dinner|
|     39.42|7.58|  Male|    No|Sat|Dinner|
|      21.7| 4.3|  Male|    No|Sat|Dinner|
+----------+----+------+------+---+------+
only showing top 10 rows


# Ejercicio 3: Agrupación con groupBy

## Objetivo
Usar `groupBy` y funciones de agregación.

## Instrucciones
Agrupa el dataset por `day` y calcula:

1. Total de cuentas por día
2. Promedio de `total_bill`
3. Promedio de `tip`
4. Máxima cuenta (`max(total_bill)`)

Ordena el resultado por el promedio de cuenta de mayor a menor.

## Pregunta
¿Qué día tiene el mayor promedio de consumo?

# Ejercicio 4: Crear un UDF personalizado

## Objetivo
Aplicar una función personalizada sobre una columna del DataFrame.

## Instrucciones
Crea una función que clasifique el nivel de propina según el valor de `tip`:

- `Baja` si tip < 2
- `Media` si tip >= 2 y tip < 4
- `Alta` si tip >= 4

Luego:

1. Convierte esa función en un UDF
2. Crea una nueva columna llamada `tip_level`
3. Muestra:
   - total_bill
   - tip
   - tip_level

## Pregunta
¿Qué nivel de propina aparece con mayor frecuencia?

# Ejercicio 5: Visualización con Seaborn

## Objetivo
Visualizar los datos del dataset `tips` para identificar patrones de consumo y propinas.

## Instrucciones
Usa el dataset `tips` y realiza lo siguiente:

1. Convierte el DataFrame de PySpark a Pandas.
2. Crea un gráfico de tipo **boxplot**:
   - eje X: `day`
   - eje Y: `total_bill`
3. Crea un gráfico de tipo **scatterplot**:
   - eje X: `total_bill`
   - eje Y: `tip`
   - color (`hue`): `smoker`
4. Crea un gráfico de tipo **histplot**:
   - variable: `total_bill`
   - con curva KDE activada

## Preguntas de análisis
- ¿Qué día tiene cuentas más altas?
- ¿Existe relación entre total de cuenta y propina?
- ¿Cómo se distribuyen las cuentas?